## Phase 2 — Molecular Featurization & Weighted Loaders

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch.utils.data import WeightedRandomSampler
from rdkit import Chem
from rdkit.Chem import AllChem
import esm
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# --- CONFIGURATION ---
INPUT_FILE = "unified_dta_competition_split.csv"
OUTPUT_DIR = "processed_data/"
ESM_MODEL_NAME = "esm2_t33_650M_UR50D"

In [3]:
# --- HELPER FUNCTIONS FOR MOLECULE GRAPH ---
def one_hot_encoding(value, choices):
    encoding = [0] * (len(choices) + 1)
    if value in choices:
        encoding[choices.index(value)] = 1
    else:
        encoding[-1] = 1 
    return encoding

def get_atom_features(atom):
    features = []
    features += one_hot_encoding(atom.GetSymbol(), ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P']) 
    features += one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5, 6]) 
    features += one_hot_encoding(atom.GetFormalCharge(), [-1, 0, 1]) 
    features += [1 if atom.GetIsAromatic() else 0]
    features += one_hot_encoding(str(atom.GetHybridization()), ['SP', 'SP2', 'SP3', 'SP3D', 'SP3D2'])
    return features 

def get_bond_features(bond):
    features = []
    features += one_hot_encoding(str(bond.GetBondType()), ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'])
    features += [1 if bond.GetIsConjugated() else 0]
    features += [1 if bond.IsInRing() else 0]
    return features 

def smiles_to_graph_and_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None, None, None, None
        
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048) 
    fp_tensor = torch.tensor(list(fp), dtype=torch.float)

    node_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_features, dtype=torch.float) 

    edges_list = []
    edge_features_list = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        
        edges_list.append((i, j))
        edge_features_list.append(bond_feat)
        edges_list.append((j, i))
        edge_features_list.append(bond_feat)

    if len(edges_list) > 0:
        edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous() 
        edge_attr = torch.tensor(edge_features_list, dtype=torch.float) 
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 7), dtype=torch.float)

    return x, edge_index, edge_attr, fp_tensor

# --- ESM-2 PROTEIN EMBEDDING ---
def compute_protein_embeddings(unique_sequences, device):
    print(f"Loading ESM-2 model: {ESM_MODEL_NAME}...")
    model, alphabet = esm.pretrained.load_model_and_alphabet(ESM_MODEL_NAME)
    model = model.to(device)
    model.eval()
    batch_converter = alphabet.get_batch_converter()
    
    embeddings_dict = {}
    print(f"Extracting representations for {len(unique_sequences)} unique proteins...")
    with torch.no_grad():
        for seq in tqdm(unique_sequences):
            trunc_seq = seq[:1022] 
            data = [("protein", trunc_seq)]
            _, _, batch_tokens = batch_converter(data)
            batch_tokens = batch_tokens.to(device)
            
            results = model(batch_tokens, repr_layers=[model.num_layers], return_contacts=False)
            token_representations = results["representations"][model.num_layers]
            sequence_rep = token_representations[0, 1 : len(trunc_seq) + 1].mean(0)
            embeddings_dict[seq] = sequence_rep.cpu()
            
    return embeddings_dict

In [4]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
# Load Dataset
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

Loading unified_dta_competition_split.csv...


In [7]:
# Pre-compute protein embeddings
unique_proteins = df['protein_seq'].unique()
protein_embeddings = compute_protein_embeddings(unique_proteins, device)

Loading ESM-2 model: esm2_t33_650M_UR50D...
Extracting representations for 428 unique proteins...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 428/428 [10:27<00:00,  1.47s/it]


In [8]:
# Build PyG data objects
train_data, val_data, test_data = [], [], []
    
print("Building PyTorch Geometric graphs and assembling dataset...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    smiles = row['smiles']
    seq = row['protein_seq']
    affinity = row['affinity']
    split = row['Data_Split']
        
    x, edge_index, edge_attr, fp = smiles_to_graph_and_fp(smiles)
    if x is None: continue 
            
    target_emb = protein_embeddings[seq]
    
    y_val = [affinity] if pd.notnull(affinity) else [0.0]
        
    data = Data(
        x=x, 
        edge_index=edge_index, 
        edge_attr=edge_attr, 
        fp=fp.unsqueeze(0), 
        target_emb=target_emb.unsqueeze(0), 
        y=torch.tensor(y_val, dtype=torch.float),
        in_A=torch.tensor([row['in_A']], dtype=torch.float),
        in_B=torch.tensor([row['in_B']], dtype=torch.float),
        in_C=torch.tensor([row['in_C']], dtype=torch.float),
        sample_id=row['id'], # Keep ID for final submission file
        sample_weight=torch.tensor([row['sample_weight']], dtype=torch.float)
    )
        
    if split == 'Train': train_data.append(data)
    elif split == 'Validation': val_data.append(data)
    else: test_data.append(data)

Building PyTorch Geometric graphs and assembling dataset...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48963/48963 [01:18<00:00, 621.78it/s]


In [9]:
# Save processed datasets to disk
print("Saving processed data to disk...")
torch.save(train_data, os.path.join(OUTPUT_DIR, "train_data.pt"))
torch.save(val_data, os.path.join(OUTPUT_DIR, "val_data.pt"))
torch.save(test_data, os.path.join(OUTPUT_DIR, "test_data.pt"))

Saving processed data to disk...


In [12]:
print(len(train_data))
print(len(val_data))
print(len(test_data))

23194
2578
23191


In [13]:
# Build pyg dataLoaders with weighted sampling
BATCH_SIZE = 128

print("Building PyTorch Geometric DataLoaders...")

# Extract weights for the sampler
train_weights = [data.sample_weight.item() for data in train_data]
sampler = WeightedRandomSampler(
    weights=train_weights, 
    num_samples=len(train_weights), 
    replacement=True
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print("\n--- Phase 2 Complete ---")
print(f"Train graphs: {len(train_data)} | Batches: {len(train_loader)}")
print(f"Validation graphs: {len(val_data)} | Batches: {len(val_loader)}")
print(f"Test graphs: {len(test_data)} | Batches: {len(test_loader)}")

Building PyTorch Geometric DataLoaders...

--- Phase 2 Complete ---
Train graphs: 23194 | Batches: 181
Validation graphs: 2578 | Batches: 21
Test graphs: 23191 | Batches: 182


In [14]:
for batch in train_data:
    break

In [15]:
batch

Data(x=[26, 29], edge_index=[2, 58], edge_attr=[58, 7], y=[1], fp=[1, 2048], target_emb=[1, 1280], in_A=[1], in_B=[1], in_C=[1], sample_id='train_22215', sample_weight=[1])

In [29]:
# --- Sanity Check: query batch composition ---
print("Querying Train Batch Compositions...\n")

# How many batches do you want to inspect?
num_batches_to_check = 3

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= num_batches_to_check:
        break
        
    print(f"=== Batch {batch_idx + 1} ===")
    print(f"Total graphs in batch: {batch.num_graphs}")
    
    # count each split
    count_A = int(batch.in_A.sum().item())
    count_B = int(batch.in_B.sum().item())
    count_C = int(batch.in_C.sum().item())
    
    print(f"\nBroad Representation:")
    print(f"- Contains Split A: {count_A} samples")
    print(f"- Contains Split B: {count_B} samples")
    print(f"- Contains Split C: {count_C} samples")
    
    # intersection count
    combinations = torch.stack([batch.in_A, batch.in_B, batch.in_C], dim=1)
    unique_combos, counts = torch.unique(combinations, dim=0, return_counts=True)
    
    print("\nExact Split Memberships in this batch:")
    for combo, count in zip(unique_combos, counts):
        combo_str = []
        if combo[0] == 1: combo_str.append('A')
        if combo[1] == 1: combo_str.append('B')
        if combo[2] == 1: combo_str.append('C')
        
        combo_name = ",".join(combo_str)
        percentage = (count.item() / batch.num_graphs) * 100
        
        print(f"  [{combo_name}]: {count.item():>3} samples ({percentage:>4.1f}%)")
    print("\n")

Querying Train Batch Compositions...

=== Batch 1 ===
Total graphs in batch: 128

Broad Representation:
- Contains Split A: 62 samples
- Contains Split B: 64 samples
- Contains Split C: 128 samples

Exact Split Memberships in this batch:
  [C]:  33 samples (25.8%)
  [B,C]:  33 samples (25.8%)
  [A,C]:  31 samples (24.2%)
  [A,B,C]:  31 samples (24.2%)


=== Batch 2 ===
Total graphs in batch: 128

Broad Representation:
- Contains Split A: 63 samples
- Contains Split B: 60 samples
- Contains Split C: 128 samples

Exact Split Memberships in this batch:
  [C]:  35 samples (27.3%)
  [B,C]:  30 samples (23.4%)
  [A,C]:  33 samples (25.8%)
  [A,B,C]:  30 samples (23.4%)


=== Batch 3 ===
Total graphs in batch: 128

Broad Representation:
- Contains Split A: 77 samples
- Contains Split B: 65 samples
- Contains Split C: 128 samples

Exact Split Memberships in this batch:
  [C]:  23 samples (18.0%)
  [B,C]:  28 samples (21.9%)
  [A,C]:  40 samples (31.2%)
  [A,B,C]:  37 samples (28.9%)


